# Seed variability

Trains the selected configuration (YOLOv26l, online augmentation, no offline
augmentation) with one random seed. Repeat with seeds 0 to 9, one per session,
and run the summary cell when all seeds are done.

## Setup

Select a GPU runtime (**Runtime > Change runtime type**). Outputs are written
to Google Drive, so that a run interrupted by the end of the session can be
resumed. The Roboflow API key is read from the Colab secret
`ROBOFLOW_API_KEY` (key icon in the left panel).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/hugocesarleal/grape-segmentation-training /content/grape-segmentation-training
%cd /content/grape-segmentation-training
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
OUTPUT = "/content/drive/MyDrive/grape-segmentation"
os.makedirs(OUTPUT, exist_ok=True)
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

## Dataset

Version 14 is the export without offline augmentation. The step also converts
the instances exported as boxes into polygons.

In [ ]:
VERSION = 14  # @param {type:"integer"}
!python -m grapeseg.data --version $VERSION --out /content/dataset

## Training

In [ ]:
SEED = 0  # @param {type:"integer"}
BATCH = 8  # @param {type:"integer"}
!python seed_variability.py train --data /content/dataset/data.yaml --seed $SEED --batch $BATCH --project $OUTPUT/seeds

## Resume an interrupted run

Run the setup and dataset cells again, then this cell with the same seed.

In [ ]:
!python seed_variability.py train --seed $SEED --batch $BATCH --project $OUTPUT/seeds --resume

## Summary

Mean and sample standard deviation of each metric over the available seeds.

In [ ]:
!python seed_variability.py summary --results $OUTPUT/seeds